In [ ]:
# đọc dữ liệu đã được xử lý cho 4 tập: train, test, meta_train và valid
import pandas as pd
data_train_merged = pd.read_parquet(r"merged-scaled-data\data_train_merged_robust.parquet", engine='pyarrow')
data_test_merged = pd.read_parquet(r"merged-scaled-data\data_test_merged_robust.parquet", engine='pyarrow')
data_train_meta_merged = pd.read_parquet(r"merged-scaled-data\data_train_meta_merged_robust.parquet", engine='pyarrow')
data_train_meta_merged = pd.read_parquet(r"merged-scaled-data\data_train_meta_merged_robust.parquet", engine='pyarrow')
data_valid_merged = pd.read_parquet(r"merged-scaled-data\data_valid_base_merged_robust.parquet", engine='pyarrow')

In [2]:
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
import numpy as np
import pandas as pd
import gc
import os

# copy dữ liệu để xử lý
train_df = data_train_merged.copy()
test_df = data_test_merged.copy()
valid_df = data_valid_merged.copy()
train_meta_df = data_train_meta_merged.copy()


# bỏ đi các cột chỉ có 1 giá trị duy nhất (không tính NaN)
constant_cols = [
    c for c in train_df.columns
    if c != "label" and train_df[c].nunique(dropna=False) <= 1
]
if constant_cols:
    print("Drop constant columns:", constant_cols)
    train_df.drop(columns=constant_cols, inplace=True)
    test_df.drop(columns=[c for c in constant_cols if c in test_df.columns], inplace=True)
    valid_df.drop(columns=[c for c in constant_cols if c in valid_df.columns], inplace=True)
    train_meta_df.drop(columns=[c for c in constant_cols if c in train_meta_df.columns], inplace=True)

# ép các cột thời lượng về numeric
num_recover = [c for c in ["delta_start", "handshake_duration"] if c in train_df.columns]
for col in num_recover:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce").fillna(0.0).astype("float32")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce").fillna(0.0).astype("float32")
    valid_df[col] = pd.to_numeric(valid_df[col], errors="coerce").fillna(0.0).astype("float32")
    train_meta_df[col] = pd.to_numeric(train_meta_df[col], errors="coerce").fillna(0.0).astype("float32")

# scale bằng QuantileTransformer cho các cột vừa chuyển sang numeric
if num_recover:
    recover_scaler = QuantileTransformer(output_distribution='normal', random_state=42)
    train_df[num_recover] = recover_scaler.fit_transform(train_df[num_recover])
    test_df[num_recover] = recover_scaler.transform(test_df[num_recover])
    valid_df[num_recover] = recover_scaler.transform(valid_df[num_recover])
    train_meta_df[num_recover] = recover_scaler.transform(train_meta_df[num_recover])


# chuyển label về numeric bằng label encoder
if "label" not in train_df.columns:
    raise ValueError("Không tìm thấy cột 'label' trong train_df.")

le = LabelEncoder()
train_df["label"] = le.fit_transform(train_df["label"].astype(str))

# loại bỏ các dòng trong tập còn lại có label chưa từng xuất hiện trong train

known_labels = set(le.classes_)
test_label_str = test_df["label"].astype(str)
mask_known = test_label_str.isin(known_labels)
if (~mask_known).any():
    print(f"Loại bỏ {(~mask_known).sum()} dòng test có label chưa từng xuất hiện trong train.")
    test_df = test_df.loc[mask_known].copy()
    test_label_str = test_df["label"].astype(str)
test_df["label"] = le.transform(test_label_str)


valid_label_str = valid_df["label"].astype(str)
mask_valid_known = valid_label_str.isin(known_labels)
if (~mask_valid_known).any():
    print(f"Loại bỏ {(~mask_valid_known).sum()} dòng valid có label chưa từng xuất hiện trong train.")
    valid_df = valid_df.loc[mask_valid_known].copy()
    valid_label_str = valid_df["label"].astype(str)
valid_df["label"] = le.transform(valid_label_str)


train_meta_label_str = train_meta_df["label"].astype(str)
mask_train_meta_known = train_meta_label_str.isin(known_labels)
if (~mask_train_meta_known).any():
    print(f"Loại bỏ {(~mask_train_meta_known).sum()} dòng train_meta có label chưa từng xuất hiện trong train.")
    train_meta_df = train_meta_df.loc[mask_train_meta_known].copy()
    train_meta_label_str = train_meta_df["label"].astype(str)
train_meta_df["label"] = le.transform(train_meta_label_str)


# đồng bộ thứ tự cột trên test, valid, train_meta theo train
test_df = test_df[train_df.columns]
valid_df = valid_df[train_df.columns]
train_meta_df = train_meta_df[train_df.columns]

# lưu lại file đã được xử lý
os.makedirs("merged-scaled-data", exist_ok=True)
'''
train_out = r"merged-scaled-data\data_train_final_dl.parquet"
test_out  = r"merged-scaled-data\data_test_final_dl.parquet"

train_df.to_parquet(train_out, index=False, engine="pyarrow")
test_df.to_parquet(test_out, index=False, engine="pyarrow")
'''
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
print("Valid shape:", valid_df.shape)
print("Train_meta shape:", train_meta_df.shape)
print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))
#print("Saved:", train_out)
#print("Saved:", test_out)

#del train_df, test_df
gc.collect()

<>:87: SyntaxWarning: invalid escape sequence '\d'
<>:87: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Admin\AppData\Local\Temp\ipykernel_9020\2132839213.py:87: SyntaxWarning: invalid escape sequence '\d'
  '''


Drop constant columns: ['protocol', 'active_skewness', 'active_cov', 'urg_flag_counts', 'ece_flag_counts', 'cwr_flag_counts', 'fwd_urg_flag_counts', 'fwd_ece_flag_counts', 'fwd_cwr_flag_counts', 'bwd_urg_flag_counts', 'bwd_ece_flag_counts', 'bwd_cwr_flag_counts', 'urg_flag_percentage_in_total', 'ece_flag_percentage_in_total', 'cwr_flag_percentage_in_total', 'fwd_urg_flag_percentage_in_total', 'fwd_ece_flag_percentage_in_total', 'fwd_cwr_flag_percentage_in_total', 'bwd_urg_flag_percentage_in_total', 'bwd_ece_flag_percentage_in_total', 'bwd_cwr_flag_percentage_in_total', 'fwd_urg_flag_percentage_in_fwd_packets', 'fwd_ece_flag_percentage_in_fwd_packets', 'fwd_cwr_flag_percentage_in_fwd_packets', 'bwd_urg_flag_percentage_in_bwd_packets', 'bwd_ece_flag_percentage_in_bwd_packets', 'bwd_cwr_flag_percentage_in_bwd_packets']
Train shape: (1900502, 320)
Test shape : (760207, 320)
Valid shape: (570149, 320)
Train_meta shape: (570154, 320)
Label mapping: {'APACHE-MHDDoS': np.int64(0), 'BYPASS-MHDD

0

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from tqdm.notebook import tqdm

# hàm xây dựng graph data cho GAT
# time_thresh: ngưỡng thời gian để xác định nối cạnh giữa các node (gói tin)
# max_neighbors: giới hạn số lượng cạnh được nối đối với 1 node
def build_graph_data(df, time_thresh=0.5, max_neighbors=5):

    print("Chuyển đổi timestamp sang số thực (Unix time) để tính toán.")
    # đổi timestamp sang số thực để tính toán khoảng cách giữa các gói tin
    if 'timestamp_num' not in df.columns:
        df['timestamp_num'] = pd.to_datetime(df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
    
    # sắp xếp lại dataframe để đảm bảo thứ tự thời gian đúng
    print("2. Sắp xếp lại dataframe theo thời gian...")
    df = df.sort_values(by='timestamp_num').reset_index(drop=True)
    
    # khởi tạo danh sách để lưu trữ cạnh
    edges_src = []
    edges_dst = []
    
    # nối các cạnh cùng src_ip và nhỏ hơn time_thresh giây, đồng thời giới hạn số lượng cạnh lân cận
    print(f"Lập edge_index (Cùng src_ip, <= {time_thresh}s, max {max_neighbors} edges/node)")
    
    # group dataframe theo src_ip để tìm các flow cùng src_ip
    for _, group in tqdm(df.groupby('src_ip', observed=True), desc="Nối cạnh src_ip"):
        idx = group.index.values # lấy index của các dòng trong group
        t = group['timestamp_num'].values # lấy timestamp đã chuyển đổi sang số thực để tính toán khoảng cách thời gian giữa các gói tin
        
        # searchsorted: tìm vị trí chèn để giữ thứ tự sắp xếp, giúp xác định nhanh các gói tin tiếp theo trong cửa sổ time_thresh
        right_bounds = np.searchsorted(t, t + time_thresh, side='right')
        
        # giới hạn số lượng cạnh lân cận bằng max_neighbors
        for i in range(len(idx)):
            limit = min(right_bounds[i], i + 1 + max_neighbors)

            # nối cạnh giữa node i và các node j trong cửa sổ thời gian
            for j in range(i + 1, limit):
                edges_src.append(idx[i])
                edges_dst.append(idx[j])

    # nối các cạnh cùng dst_ip và nhỏ hơn time_thresh giây, đồng thời giới hạn số lượng cạnh lân cận
    print(f"Lập edge_index (Cùng dst_ip, <= {time_thresh}s, max {max_neighbors} edges/node)")
    for _, group in tqdm(df.groupby('dst_ip', observed=True), desc="Nối cạnh dst_ip"):
        idx = group.index.values
        t = group['timestamp_num'].values
        
        right_bounds = np.searchsorted(t, t + time_thresh, side='right')
        
        for i in range(len(idx)):
            limit = min(right_bounds[i], i + 1 + max_neighbors)
            for j in range(i + 1, limit):
                edges_src.append(idx[i])
                edges_dst.append(idx[j])
    
    # hợp nhất danh sách nối và xóa các cạnh trùng nhau
    print("Hợp nhất danh sách vòng nối (Undirected Graph) và xóa trùng lặp...")
    if len(edges_src) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        all_src = edges_src + edges_dst
        all_dst = edges_dst + edges_src
        
        edge_index_np = np.array([all_src, all_dst], dtype=np.int64)
        
        # xóa các cạnh trùng nhau bằng cách sử dụng np.unique trên cột (src, dst)
        edge_index_np = np.unique(edge_index_np, axis=1)
        edge_index = torch.tensor(edge_index_np, dtype=torch.long).contiguous() # Cần contiguous
    
    # lấy node features và label
    print("Trích xuất Node Features (X) và Label (y)...")
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    cols_present = [c for c in cols_to_drop if c in df.columns]
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    
    # khởi tạo mảng numpy float32 và load từng cột vào để tránh tạo ra mảng float64 gây tràn RAM
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
        
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()
    
    # in ra thông tin về số lượng node và cạnh sau khi hoàn thiện graph
    print(f"HOÀN THIỆN PYG DATA OBJECT!")
    print(f"   - Số Node (Dòng) : {x_tensor.size(0):,}")
    print(f"   - Cạnh (Đã x2)   : {edge_index.size(1):,}")
    
    return Data(x=x_tensor, edge_index=edge_index, y=y_tensor)

In [ ]:
# xây dựng đồ thị cho tập train_base và valid_base (2 đồ thị này riêng biệt)
print("Đang dựng Graph cho Train Base...")
graph_train_base = build_graph_data(train_df, time_thresh=0.5)

print("\nĐang dựng Graph cho Valid Base...")
graph_valid_base = build_graph_data(valid_df, time_thresh=0.5)


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv  # <-- DÙNG GATv2 thay vì GATv1 để tránh "hội tụ tĩnh"
from torch_geometric.utils import dropout_edge
import torch.nn as nn
from tqdm.notebook import tqdm
import copy

# tạo dataloader NeighborLoader cho tập train và valid
def create_dataloader(graph_data, batch_size=2048, shuffle=True):
    print(f"Đang khởi tạo NeighborLoader (Kích thước Batch: {batch_size}, Shuffle: {shuffle})...")
    loader = NeighborLoader(
        graph_data,
        # Giảm số lượng neighbor lấy mấu để chống Phình to Đồ thị con (Graph Explosion)
        num_neighbors=[10, 5], 
        batch_size=batch_size,
        shuffle=shuffle, 
        num_workers=0 
    )
    return loader

# xây dựng model GAT với 2 lớp, dùng Multi-Head Attention + Dropout
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout # tỷ lệ loại bỏ ngẫu nhiên các cạnh trong đồ thị
        
        # Lớp GAT 1: Dùng GATv2Conv (Universal Attention) giúp model không bị bó hẹp sự chú ý
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1)  # Giảm Dropout
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads) # Kìm hãm sự sai số của các Feature 
        
        # Lớp GAT 2: Nén lại
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1)  # Giảm Dropout
        self.bn2 = nn.BatchNorm1d(embedding_dim) # Tạo ra Embedding ổn định
        
        # lớp phân loại cuối cùng: Đẩy embedding thu gọn xuống số lớp cần phân loại
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index):

        # dropout_edge
        edge_index_dp, _ = dropout_edge(
            edge_index, 
            p=self.edge_dropout, 
            force_undirected=True, 
            training=self.training
        )
        
        x = F.dropout(x, p=0.1, training=self.training)  # Giảm feature dropout
        
        x = self.conv1(x, edge_index_dp)
        x = self.bn1(x) # Chèn BatchNorm1D
        x = F.elu(x)
        
        edge_index_dp2, _ = dropout_edge(
            edge_index, 
            p=self.edge_dropout, 
            force_undirected=True, 
            training=self.training
        )
        
        x = F.dropout(x, p=0.1, training=self.training)  # Giảm feature dropout
        embedding = self.conv2(x, edge_index_dp2)
        embedding = self.bn2(embedding) # Chèn BatchNorm1D
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

# hàm train GAT với Early Stopping và lưu lại weights tốt nhất dựa trên validation loss
def train_gat(model, train_loader, valid_loader, device, epochs=5, lr=0.005, class_weights=None, patience=7):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4) 
    
    # Ép GAT phải bị trừ điểm rất đau nếu đoán sai class 4, 6, 9
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()
        
    best_val_loss = float('inf')
    best_model_weights = None
    no_improve_epochs = 0
    
    for epoch in range(epochs):
        # training phase
        model.train()
        total_train_loss = 0
        total_train_acc = 0
        total_train_nodes = 0
        
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            out, _ = model(batch.x, batch.edge_index)
            batch_size = batch.batch_size
            
            loss = criterion(out[:batch_size], batch.y[:batch_size])
            loss.backward()
            
            # CHỐNG NỔ GRADIENT (Gradient Clipping): Cắt ngắn các bước nhảy quá dài để không bị phá vỡ mô hình
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            
            optimizer.step()
            
            pred = out[:batch_size].argmax(dim=1)
            correct = (pred == batch.y[:batch_size]).sum().item()
            
            total_train_loss += loss.item() * batch_size
            total_train_acc += correct
            total_train_nodes += batch_size
            
            pbar_train.set_postfix({'Loss': f"{loss.item():.4f}", 'Acc': f"{correct/batch_size:.4f}"})
            
        avg_train_loss = total_train_loss / total_train_nodes
        avg_train_acc = total_train_acc / total_train_nodes
        
        # validation phase
        model.eval()
        total_val_loss = 0
        total_val_acc = 0
        total_val_nodes = 0
        
        with torch.no_grad():
            pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
            for batch in pbar_val:
                batch = batch.to(device)
                out, _ = model(batch.x, batch.edge_index)
                batch_size = batch.batch_size
                
                loss = criterion(out[:batch_size], batch.y[:batch_size])
                pred = out[:batch_size].argmax(dim=1)
                correct = (pred == batch.y[:batch_size]).sum().item()
                
                total_val_loss += loss.item() * batch_size
                total_val_acc += correct
                total_val_nodes += batch_size
                
                pbar_val.set_postfix({'Loss': f"{loss.item():.4f}", 'Acc': f"{correct/batch_size:.4f}"})
        
        # Thêm 2 dòng lệnh "Dọn Rác Tranh Thủ" (GC) vào sau mỗi Epoch để dọn VRAM/RAM
        import gc
        gc.collect()
        torch.cuda.empty_cache()
                
        avg_val_loss = total_val_loss / total_val_nodes
        avg_val_acc = total_val_acc / total_val_nodes
        
        print(f"=== Epoch {epoch+1} Tổng kết ===")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Acc: {avg_train_acc:.4f}")
        print(f"   Valid Loss: {avg_val_loss:.4f}   | Valid Acc: {avg_val_acc:.4f}")
        
        # early stopping: nếu validation loss tốt hơn kỷ lục, lưu lại weights tốt nhất
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            no_improve_epochs = 0
            # Copy riêng cấu trúc weights ra để không bị ghi đè chéo (Deepcopy)
            best_model_weights = copy.deepcopy(model.state_dict())
            print(f"Best model. Đã lưu trọng số tốt nhất tại Epoch {epoch+1} (Val Loss: {best_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"Báo động Early Stopping: Không cải thiện {no_improve_epochs}/{patience} epochs.")
            if no_improve_epochs >= patience:
                print(f"\n🛑 Đã kích hoạt Early Stopping tại Epoch {epoch+1}! Mô hình không cải thiện. Dừng huấn luyện.")
                break

    # phục hồi lại weights tốt nhất sau khi training hoàn tất để dùng trích xuất embedding cho XGBoost
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nHoàn tất Train! Đã load lại weights tốt nhất (Val Loss: {best_val_loss:.4f}) để dùng trích xuất Embedding!")
        
    return model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Đang Train trên:", device)

# tạo dataloader cho train và valid để train
train_loader = create_dataloader(graph_train_base, batch_size=8192, shuffle=True)
valid_loader = create_dataloader(graph_valid_base, batch_size=8192, shuffle=False)

# số feature cho trong graph_train_base
num_features = graph_train_base.x.shape[1] 

# số lượng class
num_classes = len(known_labels) if 'known_labels' in locals() else len(torch.unique(graph_train_base.y))
print(f"Số lượng Classes (Loại tấn công): {num_classes}")

# khởi tạo model GAT
model = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=64,      # Mở rộng khả năng đại diện ẩn (trước là 32)
     embedding_dim=64,        # Tăng mạnh Bandwidth thông tin truyền sang XGBoost (trước là 16)
     num_classes=num_classes, 
     heads=8,                 # Đa dạng hóa các góc nhìn Giao thức/Kết nối (trước là 4)
     edge_dropout=0.15        # Giảm hụt kết nối (trước là 0.25)
).to(device)

model = train_gat(model, train_loader, valid_loader, device, epochs=10, lr=0.01) # train gat với 10 epochs, learning rate 0.01

In [6]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

# hàm lấy embedding từ GAT đã train để làm input cho XGBoost
@torch.no_grad() # tắt tính đạo hàm
def extract_embeddings(model, graph_data, batch_size=8192, device='cpu'):
    model.eval() # chuyển model sang chế độ evaluation
    print("Đang khởi tạo Inference Loader để lấy Embeddings...")
    
    # Dùng neighbor loader với cùng cấu hình như khi train để đảm bảo trích xuất embedding đúng cách, nhưng shuffle=False để giữ thứ tự dữ liệu
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[15, 10], 
        batch_size=batch_size,
        shuffle=False, 
        num_workers=0
    )
    
    # lưu tất cả embedding và label vào mảng Python để sau này nối lại thành ma trận lớn
    all_embeddings = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ")
    for batch in pbar:
        batch = batch.to(device)
        
        # lấy embedding từ model, bỏ qua phần output phân loại cuối cùng, chỉ lấy phần embedding thu gọn từ lớp GAT thứ 2
        _, emb = model(batch.x, batch.edge_index)
        
        # lấy batch_size thực tế (có thể nhỏ hơn batch_size nếu là batch cuối cùng) để chỉ lấy phần embedding và label tương ứng
        bs = batch.batch_size
        
        # lấy embedding và label của batch hiện tại, chuyển về CPU và numpy để lưu vào list
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
    # nối tất cả embedding và label lại thành ma trận lớn để dùng cho XGBoost
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

# huấn luyện và đánh giá mô hình XGBoost sử dụng embedding từ GAT làm input
def train_evaluate_xgboost(X_train_emb, y_train, X_test_emb, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST ---")
    
    # tính toán Sample Weights để cân bằng lại Label (Trọng số lớn cho class thiểu số)
    weights = compute_sample_weight(class_weight='balanced', y=y_train)
    
    # khởi tạo mô hình XGBoost (thêm các tham số chống overfitting và điều chỉnh để phù hợp với embedding)
    xgb_model = xgb.XGBClassifier(
        n_estimators=150,      
        learning_rate=0.05,    # Giảm LR để học chậm rải đều
        max_depth=4,           # Giảm độ sâu của cây để không học vẹt các IP cũ (trước là 6)
        min_child_weight=3,    # Tránh chia tách lá nếu quá ít dữ liệu
        reg_lambda=2.0,        # L2 Regularization (Penalize weights)
        reg_alpha=0.5,         # L1 Regularization (Feature selection)
        subsample=0.8,         # Lấy mẫu 80% dữ liệu mỗi cây
        colsample_bytree=0.8,  # Lấy ngẫu nhiên 80% features (embeddings) mỗi cây
        random_state=42,
        eval_metric='mlogloss',
        tree_method='hist', device='cuda'
    )
    
    # train mô hình XGBoost
    print("Đang huấn luyện XGBoost với Class Weights... ")
    xgb_model.fit(X_train_emb, y_train, sample_weight=weights)
    print("Huấn luyện xong XGBoost!")
    
    # đánh giá mô hình trên tập test
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    return xgb_model

In [ ]:
# bắt đầu quá trình dựng graph, trích xuất embedding và huấn luyện XGBoost

# 1. Dựng Graph cho tập Train Meta (Dùng để dạy XGBoost) và tập Test (Chấm điểm cuối)
print("Đang dựng Graph cho Train Meta...")
graph_train_meta = build_graph_data(train_meta_df, time_thresh=0.5)

print("\nĐang dựng Graph cho Test...")
graph_test = build_graph_data(test_df, time_thresh=0.5)

# trích xuất embedding từ GAT đã train cho tập Train Meta (dùng để dạy XGBoost) và tập Test (dùng để chấm điểm cuối)
print("\n[GAT] Trích xuất Vector Embedding cho tập Train Meta...")
X_train_meta_emb, y_train_meta_emb = extract_embeddings(model, graph_train_meta, batch_size=8192, device=device)

print("\n[GAT] Trích xuất Vector Embedding cho tập Test...")
X_test_emb, y_test_emb = extract_embeddings(model, graph_test, batch_size=8192, device=device)

# phân loại bằng XGBoost sử dụng embedding từ GAT làm input
final_xgb = train_evaluate_xgboost(X_train_meta_emb, y_train_meta_emb, X_test_emb, y_test_emb)

# lưu lại mô hình XGBoost đã train để sử dụng cho việc dự đoán trên tập valid_base sau này
final_xgb.save_model("gat_xgb_hybrid_model.json")
print("\nĐã lưu mô hình XGBoost thành công vào file 'gat_xgb_hybrid_model.json'!")

In [ ]:
from sklearn.metrics import f1_score, classification_report
import torch
import numpy as np
from tqdm.notebook import tqdm

# nếu chưa dựng graph cho train_meta, thì dựng luôn để đánh giá thử sức mạnh phân loại của chính mạng GAT trước khi dùng XGBoost
if 'graph_test_meta' not in locals():
    print("Đang dựng Graph cho Train Meta...")
    graph_train_meta = build_graph_data(train_meta_df, time_thresh=0.5)

# hàm đánh giá trực tiếp mạng GAT trên tập Train Meta để xem khả năng phân loại của chính GAT mà không dùng XGBoost
@torch.no_grad()
def evaluate_gat_only(model, graph_data, device='cpu', batch_size=8192):
    model.eval()
    print("Đang chạy DataLoader để đánh giá trực tiếp mạng GAT...")
    
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[15, 10], 
        batch_size=batch_size,
        shuffle=False, 
        num_workers=0
    )
    
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc="[GAT] Đang Evaluation")
    for batch in pbar:
        batch = batch.to(device)
        
        # lấy giá trị out(phân loại cuối cùng) từ model GAT, bỏ qua phần embedding
        out, _ = model(batch.x, batch.edge_index)
        
        bs = batch.batch_size
        preds = out[:bs].argmax(dim=1)
        
        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
    final_preds = np.concatenate(all_preds, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    # Tính F1-Score
    f1_macro = f1_score(final_labels, final_preds, average='macro')
    f1_weighted = f1_score(final_labels, final_preds, average='weighted')
    
    print("\n============ KẾT QUẢ ĐÁNH GIÁ MẠNG GAT TRỰC TIẾP TRÊN TRAIN_META ============")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    print("Classification Report chi tiết:")
    print(classification_report(final_labels, final_preds, digits=4))
    
    return final_preds, final_labels

# tính toán và in ra kết quả đánh giá trực tiếp của mạng GAT trên tập Train Meta
gat_preds, gat_labels = evaluate_gat_only(model, graph_train_meta, device=device)

Cắt Dữ liệu theo Dòng thời gian (Time-based Masking) thay vì ngẫu nhiên để tránh Data Leakage qua cạnh nối (Neighbors Edge)

In [8]:
# đọc dữ liệu đã được xử lý cho 4 tập: train, test, meta_train và valid
import pandas as pd
data_train_merged = pd.read_parquet(r"merged-scaled-data\data_train_merged_robust.parquet", engine='pyarrow')
data_test_merged = pd.read_parquet(r"merged-scaled-data\data_test_merged_robust.parquet", engine='pyarrow')
data_train_meta_merged = pd.read_parquet(r"merged-scaled-data\data_train_meta_merged_robust.parquet", engine='pyarrow')
data_valid_merged = pd.read_parquet(r"merged-scaled-data\data_valid_base_merged_robust.parquet", engine='pyarrow')

In [20]:
import gc
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from sklearn.model_selection import train_test_split
from torch_geometric.loader import NeighborLoader
import torch
import numpy as np

# gộp 4 dataframe thành 1 dataframe duy nhất
print("Gộp 4 DataFrames thành một Siêu Dataframe duy nhất...")
df_all = pd.concat([data_train_merged, data_valid_merged, data_train_meta_merged, data_test_merged], ignore_index=True)

# xóa các dataframe gốc để giải phóng RAM, chỉ giữ lại df_all để xử lý tiếp
del data_train_merged, data_valid_merged, data_train_meta_merged, data_test_merged
gc.collect()

# xóa các cột cô định và chuẩn hóa các cột numeric để đảm bảo dữ liệu sạch sẽ trước khi dựng graph
print("Tiền xử lý (Xóa cột cố định & Chuẩn hóa Features)...")
constant_cols = [c for c in df_all.columns if c != "label" and df_all[c].nunique(dropna=False) <= 1]
if constant_cols:
    df_all.drop(columns=constant_cols, inplace=True)

num_recover = [c for c in ["delta_start", "handshake_duration"] if c in df_all.columns]
for col in num_recover:
    df_all[col] = pd.to_numeric(df_all[col], errors="coerce").fillna(0.0).astype("float32")

if num_recover:
    recover_scaler = QuantileTransformer(output_distribution='normal', random_state=42)
    df_all[num_recover] = recover_scaler.fit_transform(df_all[num_recover])

# chuyển label về numeric bằng label encoder
le = LabelEncoder()
df_all["label"] = le.fit_transform(df_all["label"].astype(str))
print("Mapping nhãn:", dict(zip(le.classes_, le.transform(le.classes_))))

print("\nSắp xếp 100% dữ liệu theo thời gian.")
if 'timestamp_num' not in df_all.columns:
    df_all['timestamp_num'] = pd.to_datetime(df_all['timestamp'], format='mixed', utc=True).astype('int64') / 10**9
df_all = df_all.sort_values(by='timestamp_num').reset_index(drop=True)


print("Tạo Graph Masks bằng Time-based Stratified Split (Chống Leakage + Cân bằng Label)...")
total_nodes = len(df_all)

# Khởi tạo ma trận mặt nạ nhị phân PyTorch (mặc định False)
train_mask = torch.zeros(total_nodes, dtype=torch.bool)
valid_mask = torch.zeros(total_nodes, dtype=torch.bool)
test_mask  = torch.zeros(total_nodes, dtype=torch.bool)

# lặp qua từng loại tấn công (Label) để cắt khúc thời gian của riêng nhãn đó
for label_val in df_all['label'].unique():
    # vì toàn bộ df_all đã được sort theo thời gian nên dòng lệnh này lấy ra các index của riêng class hiện tại mà VẪN giữ nguyên thứ tự dòng thời gian của riêng nó
    label_indices = df_all[df_all['label'] == label_val].index.values
    
    n_label = len(label_indices)
    train_end = int(n_label * 0.65)
    valid_end = int(n_label * 0.80)
    
    # Chia cắt dựa theo timeline của class hiện tại (chia tỷ lệ 65 train, 15 valid, 20 test)
    idx_train = label_indices[:train_end]
    idx_valid = label_indices[train_end:valid_end]
    idx_test  = label_indices[valid_end:]
    
    # đánh dấu True vào mặt nạ tương ứng với index đã chia cho từng tập
    train_mask[idx_train] = True
    valid_mask[idx_valid] = True
    test_mask[idx_test] = True

print(f" - Tổng Node (Gói tin): {len(df_all):,}")
print(f" - Block Train : {train_mask.sum().item():,} ({train_mask.float().mean()*100:.1f}%)")
print(f" - Block Valid : {valid_mask.sum().item():,} ({valid_mask.float().mean()*100:.1f}%)")
print(f" - Block Test  : {test_mask.sum().item():,}  ({test_mask.float().mean()*100:.1f}%)")

print("\nDựng đồ thị tổng...")
# gọi hàm xây dựng graph
super_graph = build_graph_data(df_all, time_thresh=0.5, max_neighbors=5)

# gắn mask vào đồ thị tổng để phân biệt node nào thuộc tập train, valid, test (dùng cho NeighborLoader sau này)
super_graph.train_mask = train_mask
super_graph.valid_mask = valid_mask
super_graph.test_mask  = test_mask
print("\nĐồ thị tổng đã được dựng xong, gắn Mask thành công!")

# tạo dataloader NeighborLoader cho tập train, valid, test dựa trên mask đã gắn vào đồ thị tổng
print("\nSet up Loaders Dán Nhãn (Graph Masking Loaders)...")
# Tạo Dataloader với cấu hình thu gọn để chống OOM (Out Of Memory) VRAM
train_loader = NeighborLoader(super_graph, num_neighbors=[10, 5], input_nodes=super_graph.train_mask, batch_size=2048, shuffle=True, num_workers=0)
valid_loader = NeighborLoader(super_graph, num_neighbors=[10, 5], input_nodes=super_graph.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
test_loader  = NeighborLoader(super_graph, num_neighbors=[10, 5], input_nodes=super_graph.test_mask,  batch_size=2048, shuffle=False, num_workers=0)

# xóa đi dataframe tổng để giải phóng RAM
print("\nĐang giải phóng RAM...")
del df_all 
gc.collect()
print("Hoàn tất masking theo từng nhãn và stratified time-split!")

Gộp 4 DataFrames thành một Siêu Dataframe duy nhất...
Tiền xử lý (Xóa cột cố định & Chuẩn hóa Features)...
Mapping nhãn: {'APACHE-MHDDoS': np.int64(0), 'BYPASS-MHDDoS': np.int64(1), 'CFB-MHDDoS': np.int64(2), 'DGB-MHDDoS': np.int64(3), 'HEAD-MHDDoS': np.int64(4), 'HTTP-Flood-Xerox': np.int64(5), 'KILLER-MHDDoS': np.int64(6), 'Mixed-Get-Post-Head': np.int64(7), 'NULL-MHDDoS': np.int64(8), 'POST-Flood-MHDDoS': np.int64(9), 'STOMP-MHDDoS': np.int64(10)}

Sắp xếp 100% dữ liệu theo thời gian.
Tạo Graph Masks bằng Time-based Stratified Split (Chống Leakage + Cân bằng Label)...
 - Tổng Node (Gói tin): 3,801,012
 - Block Train : 2,470,651 (65.0%)
 - Block Valid : 570,154 (15.0%)
 - Block Test  : 760,207  (20.0%)

Dựng đồ thị tổng...
Chuyển đổi timestamp sang số thực (Unix time) để tính toán.
2. Sắp xếp lại dataframe theo thời gian...
Lập edge_index (Cùng src_ip, <= 0.5s, max 5 edges/node)


Nối cạnh src_ip:   0%|          | 0/20 [00:00<?, ?it/s]

Lập edge_index (Cùng dst_ip, <= 0.5s, max 5 edges/node)


Nối cạnh dst_ip:   0%|          | 0/21 [00:00<?, ?it/s]

Hợp nhất danh sách vòng nối (Undirected Graph) và xóa trùng lặp...
Trích xuất Node Features (X) và Label (y)...
HOÀN THIỆN PYG DATA OBJECT!
   - Số Node (Dòng) : 3,801,012
   - Cạnh (Đã x2)   : 64,116,644

Đồ thị tổng đã được dựng xong, gắn Mask thành công!

Set up Loaders Dán Nhãn (Graph Masking Loaders)...

Đang giải phóng RAM...
Hoàn tất masking theo từng nhãn và stratified time-split!


In [23]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Đang Train trên:", device)
# dọn sách memory RAM trước khi train để tránh bị tràn bộ nhớ khi train GAT trên đồ thị lớn
print("Dọn dẹp RAM trước khi Train GAT...")
gc.collect()
num_features = super_graph.x.shape[1] 
# PyG tự nhận diện số lượng class duy nhất trong label (y)
num_classes = len(torch.unique(super_graph.y)) 
print(f"Số lượng Classes (Loại tấn công): {num_classes}")

# Khởi tạo mô hình GAT_Embedder đã được trang bị kỹ thuật DropEdge/Dropout
model = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=64,      
     embedding_dim=64,        
     num_classes=num_classes, 
     heads=8,
     edge_dropout=0.15        
).to(device)

# Tạo mảng Class_Weights dùng cho hàm CrossEntropyLoss của GAT
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_y_cpu = super_graph.y[super_graph.train_mask].cpu().numpy()
unique_classes_arr = np.unique(train_y_cpu)
raw_weights_arr = compute_class_weight('balanced', classes=unique_classes_arr, y=train_y_cpu)
gat_class_weights = torch.tensor(raw_weights_arr, dtype=torch.float32).to(device)
print(f"Trọng số cân bằng Class Weights GAT: {gat_class_weights}")

print("\nBắt đầu quá trình Huấn Luyện GAT với Cơ chế Masking!")
# truyền train_loader và valid_loader đã được setup với mask vào hàm train_gat để huấn luyện model GAT
model = train_gat(model, train_loader, valid_loader, device, epochs=30, lr=0.003, class_weights=gat_class_weights)

Đang Train trên: cuda
Dọn dẹp RAM trước khi Train GAT...
Số lượng Classes (Loại tấn công): 11
Trọng số cân bằng Class Weights GAT: tensor([ 3.4828,  0.1428, 27.5082,  1.9065,  3.6422, 28.2273,  5.8353,  0.6433,
         4.1272,  1.6645,  3.7230], device='cuda:0')

Bắt đầu quá trình Huấn Luyện GAT với Cơ chế Masking!


Đang Train trên: cuda
Dọn dẹp RAM trước khi Train GAT...
Số lượng Classes (Loại tấn công): 11
Trọng số cân bằng Class Weights GAT: tensor([ 3.4828,  0.1428, 27.5082,  1.9065,  3.6422, 28.2273,  5.8353,  0.6433,
         4.1272,  1.6645,  3.7230], device='cuda:0')

Bắt đầu quá trình Huấn Luyện GAT với Cơ chế Masking!


Epoch 1/30 [Train]:   0%|          | 0/1207 [00:00<?, ?it/s]

Đang Train trên: cuda
Dọn dẹp RAM trước khi Train GAT...
Số lượng Classes (Loại tấn công): 11
Trọng số cân bằng Class Weights GAT: tensor([ 3.4828,  0.1428, 27.5082,  1.9065,  3.6422, 28.2273,  5.8353,  0.6433,
         4.1272,  1.6645,  3.7230], device='cuda:0')

Bắt đầu quá trình Huấn Luyện GAT với Cơ chế Masking!


Epoch 1/30 [Train]:   0%|          | 0/1207 [00:00<?, ?it/s]

...

Epoch 12/30 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 12 Tổng kết ===
   Train Loss: 0.3362 | Train Acc: 0.9115
   Valid Loss: 0.0192   | Valid Acc: 0.9952
Báo động Early Stopping: Không cải thiện 6/7 epochs.


Epoch 13/30 [Train]:   0%|          | 0/1207 [00:00<?, ?it/s]

Epoch 13/30 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 13 Tổng kết ===
   Train Loss: 0.3327 | Train Acc: 0.9122
   Valid Loss: 0.0192   | Valid Acc: 0.9944
Báo động Early Stopping: Không cải thiện 7/7 epochs.

🛑 Đã kích hoạt Early Stopping tại Epoch 13! Mô hình không còn cải thiện. Dừng huấn luyện.

Hoàn tất Train! Đã load lại weights tốt nhất (Val Loss: 0.0126) để dùng trích xuất Embedding!


In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight
from tqdm.notebook import tqdm

@torch.no_grad()
def extract_embeddings_mask(model, graph_data, loader, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader múc lấy Embeddings...")
    
    all_embeddings = []
    all_labels = []
    
    # Do Dataloader này chỉ trả về những Node theo đúng Mask (Train/Valid/Test),
    # nên ta an tâm múc trực tiếp ra cho XGBoost
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Embedding")
    for batch in pbar:
        batch = batch.to(device)
        
        # Đẩy qua GAT, xuất Embedding (bỏ qua giá trị Output phân loại của GAT)
        _, emb = model(batch.x, batch.edge_index)
        
        # Chỉ tập trung thu Vector Embedding của các Core Nodes
        bs = batch.batch_size
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

def train_evaluate_xgboost(X_train_emb, y_train, X_valid_emb, y_valid, X_test_emb, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    # TINH CHỈNH THỦ CÔNG DỰA TRÊN KẾT QUẢ CLASSIFICATION REPORT GẦN ĐÂY:
    if 4 in weights_dict: weights_dict[4] *= 0.6
    if 6 in weights_dict: weights_dict[6] *= 1.5
    if 9 in weights_dict: weights_dict[9] *= 2.5
    if 1 in weights_dict: weights_dict[1] *= 0.8

    # Chuyển đổi thành mảng Sample Weight cho tập train
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=10,
        min_child_weight=2,
        gamma=0.4,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.9,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",
        early_stopping_rounds=40
    )
    
    print("🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("⏳ Đang huấn luyện XGBoost... (Có sử dụng Valid Set để Tự động Dừng - Early Stopping)")
    xgb_model.fit(
        X_train_emb, y_train,
        sample_weight=final_sample_weights,
        eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
        verbose=100 
    )
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK % ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

# ----------------- CHẠY THỰC THI XGBOOST -----------------
print("\n[Trích xuất cho Train Mask]")
X_train_emb, y_train_emb = extract_embeddings_mask(model, super_graph, train_loader, device=device)

print("\n[Trích xuất cho Valid Mask] - Đóng vai trò Giám khảo (MỚI)")
X_valid_emb, y_valid_emb = extract_embeddings_mask(model, super_graph, valid_loader, device=device)

print("\n[Trích xuất cho Test Mask]")
X_test_emb, y_test_emb = extract_embeddings_mask(model, super_graph, test_loader, device=device)

# BƯỚC 2: Huấn luyện và Đánh giá (Truyền thêm Valid)
hybrid_xgb = train_evaluate_xgboost(X_train_emb, y_train_emb, X_valid_emb, y_valid_emb, X_test_emb, y_test_emb)

# BƯỚC 3: Lưu Output Hybrid
hybrid_xgb.save_model("GAT_XGB_Hybrid_Masking_V3_Tuned.json")
print("\n💾 Đã lưu thành công XGBoost Object 'GAT_XGB_Hybrid_Masking_V3_Tuned.json'")

In [25]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import gc
import torch

print("\n--- BẮT ĐẦU TRAIN XGBOOST CHỈ TỪ RAW FEATURES ---")

# 1. Trích xuất trực tiếp Raw Features từ đồ thị tĩnh (không cần qua GAT hay NeighborLoader)
print("Đang trích xuất Raw Features (X) trực tiếp từ tensor bằng Mask...")
X_train_raw = super_graph.x[super_graph.train_mask].cpu().numpy()
y_train_raw = super_graph.y[super_graph.train_mask].cpu().numpy()

X_valid_raw = super_graph.x[super_graph.valid_mask].cpu().numpy()
y_valid_raw = super_graph.y[super_graph.valid_mask].cpu().numpy()

X_test_raw = super_graph.x[super_graph.test_mask].cpu().numpy()
y_test_raw = super_graph.y[super_graph.test_mask].cpu().numpy()

print(f"Kích thước X_train_raw: {X_train_raw.shape}")
print(f"Kích thước X_valid_raw: {X_valid_raw.shape}")
print(f"Kích thước X_test_raw:  {X_test_raw.shape}")

# 2. Tính toán Custom Smoothed Class Weights giống hệt cấu hình đã tune cho tập Concat
print("\nĐang tính toán Custom Smoothed Class Weights...")
unique_classes_raw = np.unique(y_train_raw)
raw_class_weights_raw = compute_class_weight('balanced', classes=unique_classes_raw, y=y_train_raw)
weights_dict_raw = {c: np.sqrt(w) for c, w in zip(unique_classes_raw, raw_class_weights_raw)}

# Dữ nguyên buff / nerf theo kinh nghiệm từ tập dữ liệu lệch
if 4 in weights_dict_raw: weights_dict_raw[4] *= 0.6
if 6 in weights_dict_raw: weights_dict_raw[6] *= 1.5
if 9 in weights_dict_raw: weights_dict_raw[9] *= 2.5
if 1 in weights_dict_raw: weights_dict_raw[1] *= 0.8

final_sample_weights_raw = np.array([weights_dict_raw[y] for y in y_train_raw])

# 3. Cấu hình XGBoost như cũ để đảm bảo tính công bằng khi so sánh
xgb_model_raw = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.08,
    max_depth=10,
    min_child_weight=2,
    gamma=0.4,
    reg_lambda=2.0,
    reg_alpha=0.5,
    subsample=0.8,         
    colsample_bytree=0.9,
    max_delta_step=3,
    random_state=42,
    eval_metric='mlogloss',
    tree_method="hist",
    device="cuda",
    early_stopping_rounds=40
)

print("🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 4. Huấn luyện
print("⏳ Đang huấn luyện XGBoost (Pure Raw Features)...")
xgb_model_raw.fit(
    X_train_raw, y_train_raw,
    sample_weight=final_sample_weights_raw,
    eval_set=[(X_train_raw, y_train_raw), (X_valid_raw, y_valid_raw)],
    verbose=100 
)
print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model_raw.best_iteration}")

# 5. Đánh giá
print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK (ONLY RAW FEATURES) ---")
y_pred_raw = xgb_model_raw.predict(X_test_raw)

acc_raw = accuracy_score(y_test_raw, y_pred_raw)
f1_macro_raw = f1_score(y_test_raw, y_pred_raw, average='macro')
f1_weighted_raw = f1_score(y_test_raw, y_pred_raw, average='weighted')

print(f"Accuracy:            {acc_raw*100:.2f}%")
print(f"F1-Score (Macro):    {f1_macro_raw * 100:.2f}%")
print(f"F1-Score (Weighted): {f1_weighted_raw * 100:.2f}%\n")

print("Classification Report:")
print(classification_report(y_test_raw, y_pred_raw, digits=4))

# 6. Lưu file
xgb_model_raw.save_model("XGBoost_Pure_RawFeatures.json")
print("\n💾 Đã lưu XGBoost Object 'XGBoost_Pure_RawFeatures.json'")


--- BẮT ĐẦU TRAIN XGBOOST CHỈ TỪ RAW FEATURES ---
Đang trích xuất Raw Features (X) trực tiếp từ tensor bằng Mask...
Kích thước X_train_raw: (2470651, 313)
Kích thước X_valid_raw: (570154, 313)
Kích thước X_test_raw:  (760207, 313)

Đang tính toán Custom Smoothed Class Weights...
🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
⏳ Đang huấn luyện XGBoost (Pure Raw Features)...
[0]	validation_0-mlogloss:1.56418	validation_1-mlogloss:1.57039
[100]	validation_0-mlogloss:0.03448	validation_1-mlogloss:0.06692
[200]	validation_0-mlogloss:0.02500	validation_1-mlogloss:0.05461
[221]	validation_0-mlogloss:0.02415	validation_1-mlogloss:0.05507
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 181

--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK (ONLY RAW FEATURES) ---
Accuracy:            93.94%
F1-Score (Macro):    80.26%
F1-Score (Weighted): 94.14%

Classification Report:
              precision    recall  f1-score   support

           0     0.8149    0.8541    0.8341     19843
         

In [26]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tqdm.notebook import tqdm
import torch

@torch.no_grad()
def extract_concat_features_mask(model, graph_data, loader, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...")
    
    all_combined = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Nối")
    for batch in pbar:
        batch = batch.to(device)
        
        # Đẩy qua GAT, xuất Embedding
        _, emb = model(batch.x, batch.edge_index)
        
        bs = batch.batch_size
        
        # Lấy raw features và embeddings của các core node trong batch
        raw_features = batch.x[:bs].cpu().numpy()
        embeddings = emb[:bs].cpu().numpy()
        
        # Nối (concatenation) theo axis=1
        combined_matrix = np.concatenate([raw_features, embeddings], axis=1)
        
        all_combined.append(combined_matrix)
        all_labels.append(batch.y[:bs].cpu().numpy())
        
    final_combined = np.concatenate(all_combined, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Nối: {final_combined.shape}")
    return final_combined, final_labels

def train_evaluate_concat_xgboost(X_train, y_train, X_valid, y_valid, X_test, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    # TINH CHỈNH THỦ CÔNG DỰA TRÊN KẾT QUẢ CLASSIFICATION REPORT GẦN ĐÂY:
    # Class 4: Precision cực thấp (35%), Recall cao (69%) -> Model dự đoán nhầm quá nhiều vào Class 4 -> Giảm trọng số
    if 4 in weights_dict: weights_dict[4] *= 0.6
    
    # Class 6: Precision 69%, Recall 46% -> Dưới trung bình -> Tăng trọng số
    if 6 in weights_dict: weights_dict[6] *= 1.5
    
    # Class 9: Precision 99%, Recall 35% -> Quá rụt rè, miss rất nhiều -> Tăng mạnh trọng số
    if 9 in weights_dict: weights_dict[9] *= 2.5
    
    # Class 1: Dòng chiếm đa số (Majority), nhường bớt sự tập trung cho lớp khác
    if 1 in weights_dict: weights_dict[1] *= 0.8

    # Chuyển đổi thành mảng Sample Weight cho tập train
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=10,            # Tăng lại lên 10 để các nhánh phân tách tốt hơn các lớp khó (4, 6, 9)
        min_child_weight=2,
        gamma=0.4,
        reg_lambda=2.0,          # Tăng L2 Regularization để chống overfit
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.9,    # Lấy nhiều đặc trưng hơn ở mỗi nhánh
        max_delta_step=3,        # Giảm từ 5 xuống 3 để tránh giật cục (nhảy hố cạn)
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",
        early_stopping_rounds=40
    )
    
    print("🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("⏳ Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)")
    xgb_model.fit(
        X_train, y_train,
        sample_weight=final_sample_weights,
        eval_set=[(X_train, y_train), (X_valid, y_valid)],
        verbose=100 
    )
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---")
    y_pred = xgb_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

# ----------------- CHẠY THỰC THI XGBOOST CONCAT -----------------
print("\n[Trích xuất cho Train Mask]")
X_train_concat, y_train_concat = extract_concat_features_mask(model, super_graph, train_loader, device=device)

print("\n[Trích xuất cho Valid Mask]")
X_valid_concat, y_valid_concat = extract_concat_features_mask(model, super_graph, valid_loader, device=device)

print("\n[Trích xuất cho Test Mask]")
X_test_concat, y_test_concat = extract_concat_features_mask(model, super_graph, test_loader, device=device)

# Huấn luyện và Đánh giá
hybrid_xgb_concat = train_evaluate_concat_xgboost(X_train_concat, y_train_concat, X_valid_concat, y_valid_concat, X_test_concat, y_test_concat)

# Lưu Output
hybrid_xgb_concat.save_model("GAT_XGB_Hybrid_Masking_V4_Concat.json")
print("\n💾 Đã lưu thành công XGBoost Object 'GAT_XGB_Hybrid_Masking_V4_Concat.json'")


[Trích xuất cho Train Mask]
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...



[Trích xuất cho Train Mask]
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/1207 [00:00<?, ?it/s]


[Trích xuất cho Train Mask]
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/1207 [00:00<?, ?it/s]

...

Trích xuất thành công! Kích thước Ma trận Nối: (2470651, 377)

[Trích xuất cho Valid Mask]
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/279 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (570154, 377)

[Trích xuất cho Test Mask]
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/372 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (760207, 377)

--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---
Đang tính toán Custom Smoothed Class Weights...
🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
⏳ Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)
[0]	validation_0-mlogloss:1.56097	validation_1-mlogloss:1.56144
[100]	validation_0-mlogloss:0.00048	validation_1-mlogloss:0.00348
[200]	validation_0-mlogloss:0.00003	validation_1-mlogloss:0.00161
[300]	validation_0-mlogloss:0.00002	validation_1-mlogloss:0.00160
[303]	validation_0-mlogloss:0.00002	validation_1-mlogloss:0.00160
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 263

--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---
Accuracy:            94.20%
F1-Score (Macro):    81.26%
F1-Score (Weighted): 94.10%

Classification Report:
              precision    recall  f1-score   support

           0     0.7747    0.9725    0.8624     19843
           1     0.9925    1.0000    0.9963    484070
      

In [27]:
# test f1 của GAT trực tiếp trên tập test_mask
@torch.no_grad()
def evaluate_gat_only(model, graph_data, device='cpu', batch_size=8192):
    model.eval()
    print("Đang chạy DataLoader để đánh giá trực tiếp mạng GAT trên Test Mask...")
    
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[15, 10], 
        batch_size=batch_size,
        shuffle=False, 
        num_workers=0,
        input_nodes=graph_data.test_mask  # Chỉ lấy những Node thuộc Test Mask
    )
    
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc="[GAT] Đang Evaluation trên Test Mask")
    for batch in pbar:
        batch = batch.to(device)
        
        out, _ = model(batch.x, batch.edge_index)
        
        bs = batch.batch_size
        preds = out[:bs].argmax(dim=1)
        
        all_preds.append(preds.cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
    final_preds = np.concatenate(all_preds, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    f1_macro = f1_score(final_labels, final_preds, average='macro')
    f1_weighted = f1_score(final_labels, final_preds, average='weighted')
    
    print("\n============ KẾT QUẢ ĐÁNH GIÁ MẠNG GAT TRỰC TIẾP TRÊN TEST MASK ============")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    print("Classification Report chi tiết:")
    print(classification_report(final_labels, final_preds, digits=4))
    
    return final_preds, final_labels

gat_test_preds, gat_test_labels = evaluate_gat_only(model, super_graph, device=device)

Đang chạy DataLoader để đánh giá trực tiếp mạng GAT trên Test Mask...


Đang chạy DataLoader để đánh giá trực tiếp mạng GAT trên Test Mask...


[GAT] Đang Evaluation trên Test Mask:   0%|          | 0/93 [00:00<?, ?it/s]

Đang chạy DataLoader để đánh giá trực tiếp mạng GAT trên Test Mask...


[GAT] Đang Evaluation trên Test Mask:   0%|          | 0/93 [00:00<?, ?it/s]


============ KẾT QUẢ ĐÁNH GIÁ MẠNG GAT TRỰC TIẾP TRÊN TEST MASK ============
F1-Score (Macro):    76.64%
F1-Score (Weighted): 93.60%

Classification Report chi tiết:
              precision    recall  f1-score   support

           0     0.7552    0.9702    0.8493     19843
           1     0.9981    1.0000    0.9991    484070
           2     0.3002    0.9928    0.4610      2513
           3     0.9987    0.8373    0.9109     36251
           4     0.3371    0.7268    0.4606     18975
           5     0.9947    0.9959    0.9953      2449
           6     0.6119    0.3300    0.4287     11844
           7     0.9999    0.9928    0.9963    107433
           8     0.8091    0.9292    0.8650     16745
           9     0.9993    0.3517    0.5202     41521
          10     0.8937    0.9994    0.9436     18563

    accuracy                         0.9362    760207
   macro avg     0.7907    0.8296    0.7664    760207
weighted avg     0.9606    0.9362    0.9360    760207



GAT with KNN